# Implementing an Arrow Flight Server

Now we have the necessary knowledge to implement an Arrow Flight Server. We will go through each service method and implement it.

For the data layer, we're using Iceberg to offload the logic of data storing, yet having a production-grade backend that is Arrow-native. We won't go into details in this course - just know it's a way of treating parquet files on object storage like a relational database. 

`<shameless self-promotion>` If you want to learn more about Iceberg - check out [Pydata London 2025 - Hands on with Apache Iceberg](https://youtu.be/XluBSLT60h8) `</shameless self-promotion>`

First, let's setup the Iceberg client and a base for the FlightServer by subclassing the generated GRPC Server - in this case `flight.FlightServerBase`. Add some boilerplate for actually running and we're good to go.

In [36]:
from pyiceberg.catalog.rest import RestCatalog
from pyarrow import flight
import threading
import polars as pl
import json

In [2]:
catalog = RestCatalog("pydata-amsterdam", uri="http://lakekeeper:8181/catalog", warehouse="default")

In [3]:
catalog.create_namespace_if_not_exists("bikeshare")

In [76]:
class FlightServer(flight.FlightServerBase):
    _instances = []
    def __init__(self, catalog: RestCatalog):
        self._catalog = catalog
        self._namespace = "bikeshare"
        self._thread = None
        self.location = "grpc://localhost:65432"
        # For demo purposes, we will create a lot of servers.
        # We need to make sure all previous instances are closed before we start a new one
        if FlightServer._instances:
            for instance in FlightServer._instances:
                instance.stop_background()
            FlightServer._instances.clear()
        
        super().__init__(location=self.location)
        FlightServer._instances.append(self)

    
    # Some helpers to enable us to start and stop the Server
    def start_background(self):

        if self._thread is not None and self._thread.is_alive():
            return
        self._thread = threading.Thread(target=self.serve, daemon=True)
        self._thread.start()
        print(f"Flight server listening at {self.location}")

    def stop_background(self):
        if self._thread is None or not self._thread.is_alive():
            print("Not running.")
            return
        self.shutdown()
        self.wait()
        self._thread.join(timeout=5)
        self._thread = None
        print("Flight server stopped.")

## Do_put - Uploading data

Next, we need to upload some data, so let's implement the `do_put` - the Protobuf looks like this:

```protobuf
message FlightData {
  FlightDescriptor flight_descriptor = 1;
  bytes data_header = 2;
  bytes app_metadata = 3;
  bytes data_body = 1000;
}

message FlightDescriptor {
  enum DescriptorType {
    PATH = 1;
    CMD = 2;
  }
  DescriptorType type = 1;
  bytes cmd = 2;
  repeated string path = 3;

message PutResult {
  bytes app_metadata = 1;
}

service FlightService {
    rpc DoPut(stream FlightData) returns (stream PutResult) {}
}
```



The first thing to notice is that it's not an exact match to our Protobuf definition, and that's because the Python Arrow Flight implementation is
backed by the C++ implementation, so it follows those signatures which deconstruct the FlightData object a bit.

We get a reader and a writer object to match the stream in and stream out in the Service definition. Then we get a new object - the `FlightDescriptor`, which has an enum for PATH and CMD. Arrow Flight is meant to be a general purpose framework, so it doesn't impose semantic definitions on us. We need to decide what PATH and CMD mean for our application, use both or none. 

### FlightDescriptor.PATH
Path is meant to describe f.ex file locations or objects, something addressable. We can send as many paths as we want

### FlightDescriptor.CMD
Semantically a CMD contains a command to be executed - to Flight it's just arbitrary bytes, so we would implement a type of command.
Could be JSON, could be Protobuf, could be anything.

In our little server, we will assume a PATH is the name of the table we want to append to.

In [77]:
class DoPutServer(FlightServer):
    def do_put(self, 
               ctx: flight.ServerCallContext, 
               descriptor: flight.FlightDescriptor, 
               reader: flight.MetadataRecordBatchReader, 
               writer: flight.FlightMetadataWriter):
        # path is a list of bytes - we assume there'll only be one path for ease of implementation
        table_name = descriptor.path[0].decode()
        
        # The reader knows the Arrow schema of its data
        table = self._catalog.create_table_if_not_exists((self._namespace, table_name), schema=reader.schema)
        
        # We can also iterate over the chunks and batch insert them
        arrow_table = reader.read_all()
        table.append(arrow_table)

        # We can send messages back to the writer
        msg = f"Inserted {len(arrow_table)} rows"
        
        # Arrow Flight expects bytes to be sent, so always encode strings
        writer.write(msg.encode())


In [83]:
server = DoPutServer(catalog=catalog)
server.start_background()

Flight server stopped.
Flight server listening at grpc://localhost:65432


We now have a running Flight Server, we can create a Flight Client

In [84]:
client = flight.connect("grpc://localhost:65432")

We're now ready to `do_put` - we have some data handy in `/app/data`

In [70]:
!ls /app/data

202601-citibike-tripdata_1.csv	campaigns.csv
202601-citibike-tripdata_2.csv	messages-demo.csv


In [71]:
table = pl.read_csv('/app/data/202601-citibike-tripdata_1.csv', schema_overrides={"start_station_id": pl.String, "end_station_id": pl.String}).to_arrow()

In [72]:
table

pyarrow.Table
ride_id: large_string
rideable_type: large_string
started_at: large_string
ended_at: large_string
start_station_name: large_string
start_station_id: large_string
end_station_name: large_string
end_station_id: large_string
start_lat: double
start_lng: double
end_lat: double
end_lng: double
member_casual: large_string
----
ride_id: [["85744AF35D7F2DF5","9D18958E5788880B","B050891B7B009EE5","0B6D7938C4EF1668","95415F60C7120CC3",...,"66326F432BAF5EB9","82EBD3ABA5235075","6C7A93BAE883AA91","C600F37F203CD508","74ADACC8272081C1"],["FAEED95167022C18","A84F319980549E8F","B2D0145E34B9C2CC","0FEDBDBEA1EB1DBA","AB65D9AEBDF78844",...,"55A7843F37F5DBD9","0DF140D135CE1289","3DAB748888D036C9","04AF59FE34562075","84B53D6B603256D5"],...,["21A108EA7B18C046","AB5EED992DF56320","004A827E442760B4","70B63F56A96DD945","85EB9592C5DF3ABD",...,"42DDA59B97421CA5","FB794EC004BA1A10","82A3DF08E0AECA85","0306C5A98896DBCD","208203ACE7E43B61"],["0D033BCE4315A989","821D5AC03ED08A7C","D8C11C663AFE486D","9E

Next, we will need a FlightDescriptor, and we will use the PATH to indicate the table to append to

In [20]:
desc = flight.FlightDescriptor.for_path("rides")

`do_put` returns the `writer` and `reader` objects where we can start streaming our data to the backend

In [21]:
writer, reader = client.do_put(desc, schema=table.schema)

stats ReadStats(num_messages=1, num_record_batches=0, num_dictionary_batches=0, num_dictionary_deltas=0, num_replaced_dictionaries=0)


In [22]:
writer.write_table(table)
writer.done_writing()

stats ReadStats(num_messages=130, num_record_batches=129, num_dictionary_batches=0, num_dictionary_deltas=0, num_replaced_dictionaries=0)


In [ ]:
msg = reader.read()
msg.to_pybytes().decode()

## Data observability - get_flight_info

Since Arrow Flight is a framework for working with datasets, it includes a number of introspection tools for the data. One of them is `get_flight_info`, which looks like this:

```protobuf
message FlightInfo {
  // We know this one
  FlightDescriptor flight_descriptor = 2;
  // Any number of FlightEndpoint - where can we fetch the data from?
  repeated FlightEndpoint endpoint = 3;
    
  // Some metadata about the dataset.
  int64 total_records = 4;
  int64 total_bytes = 5;
  // If there are multiple endpoints, will they give me ordered data?
  bool ordered = 6;
  // Arbitrary metadata goes here
  bytes app_metadata = 7;
}

message FlightEndpoint {
  // The server will hand the client a Ticket - when the client gives it back to the server, the server uses it to know what data it should fetch
  Ticket ticket = 1;
  // The Location is the url of the server where we can redeem the ticket
  // If there's more than one, client gets to pick. Edge location? lowest latency?
  repeated Location location = 2;
  // The Ticket can have an optional expiration
  google.protobuf.Timestamp expiration_time = 3;
  // Arbitrary metadata here
  bytes app_metadata = 4;
}

// A marker implemented by the server to define what data it should get
message Ticket {
  bytes ticket = 1;
}

// A URI where to send the ticket
message Location {
  string uri = 1;
}


service FlightService {
    rpc GetFlightInfo(FlightDescriptor) returns (FlightInfo) {}
}
```

In [90]:
class GetFlightInfoServer(DoPutServer):
    # Foreshadowing - we can reuse _make_flight_info 
    def _make_flight_info(self, table_name: str):
        # Grab the Iceberg table
        table = self._catalog.load_table((self._namespace, table_name))

        # Grab statistics from the Iceberg table
        snapshot = table.current_snapshot()
        num_rows = -1
        total_bytes = -1
        
        if snapshot is not None and snapshot.summary is not None:
            num_rows = int(snapshot.summary.get("total-records", -1))
            total_bytes = int(snapshot.summary.get("total-files-size", -1))

        # The ticket is whatever representation we choose - here we choose a simple JSON object
        ticket = flight.Ticket(json.dumps({"namespace": self._namespace, "table": table_name}).encode())

        # We stick to a single endpoint - each endpoint would represent a data partition
        # Arrow assumes all endpoints are consumed to get the full dataset
        endpoints = [
            flight.FlightEndpoint(
                ticket=ticket,
                locations=[self.location],
            )
        ]

        return flight.FlightInfo(
            # Our table has an Iceberg schema, but we need an Arrow schema
            schema=table.schema().as_arrow(),
            endpoints=endpoints,
            descriptor=flight.FlightDescriptor.for_path(table_name),
            total_records=num_rows,
            total_bytes=total_bytes
        )

    def get_flight_info(self, ctx: flight.ServerCallContext, descriptor: flight.FlightDescriptor) -> flight.FlightInfo:
        table_name = descriptor.path[0].decode()
        return self._make_flight_info(table_name)
        
        

In [91]:
server = GetFlightInfoServer(catalog=catalog)
server.start_background()

Flight server stopped.
Flight server listening at grpc://localhost:65432


In [92]:
client = flight.connect("grpc://localhost:65432")

In [94]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))

<pyarrow.flight.FlightInfo schema=ride_id: large_string
  -- field metadata --
  PARQUET:field_id: '1'
rideable_type: large_string
  -- field metadata --
  PARQUET:field_id: '2'
started_at: large_string
  -- field metadata --
  PARQUET:field_id: '3'
ended_at: large_string
  -- field metadata --
  PARQUET:field_id: '4'
start_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '5'
start_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '6'
end_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '7'
end_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '8'
start_lat: double
  -- field metadata --
  PARQUET:field_id: '9'
start_lng: double
  -- field metadata --
  PARQUET:field_id: '10'
end_lat: double
  -- field metadata --
  PARQUET:field_id: '11'
end_lng: double
  -- field metadata --
  PARQUET:field_id: '12'
member_casual: large_string
  -- field metadata --
  PARQUET:field_id: '13' descriptor=<pyarrow.flight.Flight

In [95]:
info.schema

ride_id: large_string
  -- field metadata --
  PARQUET:field_id: '1'
rideable_type: large_string
  -- field metadata --
  PARQUET:field_id: '2'
started_at: large_string
  -- field metadata --
  PARQUET:field_id: '3'
ended_at: large_string
  -- field metadata --
  PARQUET:field_id: '4'
start_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '5'
start_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '6'
end_station_name: large_string
  -- field metadata --
  PARQUET:field_id: '7'
end_station_id: large_string
  -- field metadata --
  PARQUET:field_id: '8'
start_lat: double
  -- field metadata --
  PARQUET:field_id: '9'
start_lng: double
  -- field metadata --
  PARQUET:field_id: '10'
end_lat: double
  -- field metadata --
  PARQUET:field_id: '11'
end_lng: double
  -- field metadata --
  PARQUET:field_id: '12'
member_casual: large_string
  -- field metadata --
  PARQUET:field_id: '13'

In [100]:
print(f"Number of records: {info.total_records:,}. \nNumber of bytes: {info.total_bytes:,}")

Number of records: 1,000,000. 
Number of bytes: 34,333,262


In [102]:
info.app_metadata

b''

In [101]:
info.endpoints

[<pyarrow.flight.FlightEndpoint ticket=<pyarrow.flight.Ticket ticket=b'{"namespace": "bikeshare", "table": "rides"}'> locations=[<pyarrow.flight.Location b'grpc://localhost:65432'>] expiration_time=None app_metadata=b''>]

## Fetching data - do_get

We actually need the FlightInfo in order to fetch data, since that's how we get a flight.Ticket - so now we are ready to implement `do_get`

```protobuf
service FlightService {
    rpc DoGet(Ticket) returns (stream FlightData) {}
}
```
No new message types - we've used them all before, we will now read a stream of FlightData instead of writing to it

In [112]:
class DoGetService(GetFlightInfoServer):
    def do_get(self, ctx: flight.ServerCallContext, ticket: flight.Ticket) -> flight.FlightDataStream:
        # We chose to use JSON as our Ticket format
        request = json.loads(ticket.ticket.decode())
        
        # Use our Ticket to load the requested data
        table = self._catalog.load_table((request["namespace"], request["table"]))

        # PyIceberg lets us stream the data as Arrow RecordBatches, exactly what Arrow Flight expects
        scanner = table.scan().to_arrow_batch_reader()
        def _gen():
            yield from scanner
        # FlightDataStream has a Generator implementation, and a RecordBatchStream implementation
        return flight.GeneratorStream(schema=table.schema().as_arrow(), generator=_gen())

In [113]:
server = DoGetService(catalog=catalog)
server.start_background()

Flight server stopped.
Flight server listening at grpc://localhost:65432


In [114]:
client = flight.connect("grpc://localhost:65432")

First we get the FlightInfo, and that will contain the Ticket we need to fetch the actual data

In [115]:
info = client.get_flight_info(flight.FlightDescriptor.for_path("rides"))
reader = client.do_get(info.endpoints[0].ticket)

Since we can see from the Protobuf definition that `do_get` returns a stream, we can iterate over the reader to get batches of data instead.
`read_all` will read the whole stream for us though.

In [117]:
df = pl.from_arrow(reader.read_all())
df

ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
str,str,str,str,str,str,str,str,f64,f64,f64,f64,str
"""85744AF35D7F2DF5""","""electric_bike""","""2026-01-02 05:36:24.539""","""2026-01-02 05:42:21.153""","""W 42 St & 8 Ave""","""6602.05""","""E 58 St & Madison Ave""","""6839.04""",40.75757,-73.990985,40.763026,-73.972095,"""member"""
"""9D18958E5788880B""","""electric_bike""","""2026-01-02 15:15:11.915""","""2026-01-02 15:18:40.462""","""Division St & Bowery""","""5311.08""","""Clinton St & Grand St""","""5303.06_""",40.71419,-73.99673,40.715738,-73.98699,"""member"""
"""B050891B7B009EE5""","""electric_bike""","""2026-01-12 10:12:51.453""","""2026-01-12 10:16:55.731""","""Broadway & 31 St""","""6789.08""","""35 Ave & 37 St""","""6563.12""",40.76194,-73.92513,40.755733,-73.923661,"""member"""
"""0B6D7938C4EF1668""","""electric_bike""","""2026-01-01 01:03:29.712""","""2026-01-01 01:05:37.341""","""34 St & 35 Ave""","""6605.08""","""35 St & Broadway""","""6750.16""",40.756933,-73.926223,40.760339,-73.922243,"""member"""
"""95415F60C7120CC3""","""electric_bike""","""2026-01-03 19:55:51.636""","""2026-01-03 20:14:57.964""","""E 6 St & Ave B""","""5584.04""","""W 55 St & 6 Ave""","""6809.09""",40.724537,-73.981854,40.763189,-73.978434,"""member"""
…,…,…,…,…,…,…,…,…,…,…,…,…
"""E771EB98769C54D0""","""electric_bike""","""2026-01-02 12:42:51.450""","""2026-01-02 12:48:28.497""","""Carlton Ave & Dean St""","""4199.12""","""Berkeley Pl & 6 Ave""","""4134.06""",40.680974,-73.97101,40.67653,-73.978469,"""member"""
"""5E1BC2953B476C78""","""electric_bike""","""2026-01-06 15:55:23.634""","""2026-01-06 16:00:08.435""","""West Thames St""","""5114.06""","""Vesey St & Church St""","""5216.06""",40.708347,-74.017134,40.71222,-74.010472,"""member"""
"""CC254688E4151B73""","""electric_bike""","""2026-01-05 22:21:08.536""","""2026-01-05 22:24:19.911""","""Broadway & Morris St""","""5033.01""","""Vesey St & Church St""","""5216.06""",40.705945,-74.013219,40.71222,-74.010472,"""member"""


# Bidirectional data exchange - do_exchange
Taking advantage of the bidirectional nature of GRPC, we can send and receive streams of data

```protobuf
service FlightServer {
    rpc DoExchange(stream FlightData) returns (stream FlightData) {}
}
```

This has a number of different uses, like calculating metrics across large amounts of data, adding context to existing data, keeping running statistics on the server side, etc.